# Práctica: Introducción a modelos de visión con *timm* y comparación de arquitecturas

## 1. Introducción

En esta práctica se introduce el uso de modelos de visión por computador preentrenados mediante la librería timm.

Se trabajará con técnicas de *transfer learning* y se compararán distintas arquitecturas en tareas de clasificación de imágenes.

La práctica se divide en:

* **Parte 1 (tutorial guiado)** → dataset sencillo
* **Parte 2 (práctica)** → problema más realista

## 2. Objetivos de aprendizaje

* Utilizar modelos preentrenados
* Adaptar modelos a nuevas tareas
* Comprender *feature extraction* vs *fine-tuning*
* Comparar arquitecturas CNN y Transformers
* Analizar el impacto del preprocesado

# 3. Parte 1: Tutorial (CIFAR-10)

## 3.1 Importaciones

In [12]:
!pip install python-dotenv

In [13]:
from dotenv import load_dotenv
import os

load_dotenv()
token = os.getenv("HF_TOKEN")

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


## 3.2 Dataset

Se utilizará CIFAR-10.

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

100%|██████████| 170M/170M [07:12<00:00, 395kB/s] 
/root/projects/computer_vision_oscar/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


## 3.3 Modelo

In [4]:
model = timm.create_model(
    "resnet18",
    pretrained=True,
    num_classes=10
).to(device)

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

# 3.4 Entrenamiento

In [5]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

def train_one_epoch():
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

def evaluate():
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

## 3.5 Ejecutar

In [6]:
for epoch in range(3):
    loss = train_one_epoch()
    acc = evaluate()
    print(f"Epoch {epoch+1} | Loss {loss:.4f} | Acc {acc:.4f}")


Epoch 1 | Loss 0.3970 | Acc 0.9133
Epoch 2 | Loss 0.1536 | Acc 0.9253
Epoch 3 | Loss 0.0977 | Acc 0.9318


## 3.6 Reflexión

* ¿Qué accuracy se obtiene?
* ¿Por qué converge rápido?
* ¿Qué aporta `pretrained=True`?

# 4. Parte 2: Práctica (Gatos vs Perros)

## 4.1 Dataset

Se utilizará el dataset Oxford-IIIT Pet.

Se transformará en un problema binario:

* 0 → gato
* 1 → perro

## 4.2 Descarga y preparación

In [7]:
from torchvision import datasets
from torch.utils.data import DataLoader

# Transformaciones base
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_full = datasets.OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="category",
    download=True,
    transform=train_tfms
)

val_full = datasets.OxfordIIITPet(
    root="./data",
    split="test",
    target_types="category",
    download=True,
    transform=val_tfms
)

# Conversión a binario
def to_binary(y):
    return 0 if y < 12 else 1

def collate_fn(batch):
    xs, ys = zip(*batch)
    xs = torch.stack(xs)
    ys = torch.tensor([to_binary(int(y)) for y in ys])
    return xs, ys

train_loader = DataLoader(train_full, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_full, batch_size=32, collate_fn=collate_fn)

100%|██████████| 792M/792M [00:09<00:00, 82.6MB/s] 
100%|██████████| 19.2M/19.2M [00:00<00:00, 44.0MB/s]


## 4.3 Modelos a comparar

* resnet18
* vit_tiny_patch16_224
* swin_tiny_patch4_window7_224

## 4.4 Funciones base

In [8]:
def create_model(name):
    return timm.create_model(name, pretrained=True, num_classes=2).to(device)

def set_trainable(model, freeze):
    if freeze:
        for p in model.parameters():
            p.requires_grad = False

        if hasattr(model, "head"):
            for p in model.head.parameters():
                p.requires_grad = True
        elif hasattr(model, "fc"):
            for p in model.fc.parameters():
                p.requires_grad = True
    else:
        for p in model.parameters():
            p.requires_grad = True


def get_optimizer(model):
    params = [p for p in model.parameters() if p.requires_grad]
    return optim.AdamW(params, lr=3e-4)

## 4.5 Entrenamiento

In [9]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

## 4.6 Experimento

In [16]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

model_name = "swin_tiny_patch4_window7_224"  # cambiar

model = create_model(model_name)

freeze = True  # probar True / False
set_trainable(model, freeze)

optimizer = get_optimizer(model)
criterion = nn.CrossEntropyLoss()



# ── Almacena métricas ────────────────────────────────────────────────────────
history = {"loss": [], "acc": []}

for epoch in range(5):
    loss = train_one_epoch(model, train_loader, optimizer, criterion)
    acc  = evaluate(model, val_loader)
    history["loss"].append(loss)
    history["acc"].append(acc)
    print(f"{model_name} | freeze={freeze} | Epoch {epoch+1} | Loss {loss:.4f} | Acc {acc:.4f}")

# ── Plot ─────────────────────────────────────────────────────────────────────
epochs = range(1, len(history["loss"]) + 1)

fig = plt.figure(figsize=(12, 5))
fig.suptitle(f"{model_name}  •  freeze={freeze}", fontsize=13, fontweight="bold")
gs  = gridspec.GridSpec(1, 2, wspace=0.35)

# — Gráfico 1: Loss —
ax1 = fig.add_subplot(gs[0])
ax1.plot(epochs, history["loss"], color="#E05C5C", linewidth=2.5,
         marker="o", markersize=6, label="Train Loss")
ax1.set_title("Loss por época", fontsize=11)
ax1.set_xlabel("Época")
ax1.set_ylabel("Loss")
ax1.set_xticks(epochs)
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.4)

# — Gráfico 2: Accuracy —
ax2 = fig.add_subplot(gs[1])
ax2.plot(epochs, history["acc"], color="#4C8EDA", linewidth=2.5,
         marker="s", markersize=6, label="Val Accuracy")
ax2.set_title("Accuracy por época", fontsize=11)
ax2.set_xlabel("Época")
ax2.set_ylabel("Accuracy")
ax2.set_xticks(epochs)
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.4)

plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


swin_tiny_patch4_window7_224 | freeze=True | Acc 0.7997


KeyboardInterrupt: 

## 4.7 Tareas

Para cada modelo:

1. Ejecutar con `freeze=True`
2. Ejecutar con `freeze=False`
3. Comparar resultados

## 4.8 Análisis

Responder:

* ¿Qué modelo funciona mejor?
* ¿Cuál converge más rápido?
* ¿Qué impacto tiene el fine-tuning?
* ¿Diferencias entre CNN y Transformers?


# 5. Parte opcional

## Objetivo

Mejorar el rendimiento modificando el preprocesado.

## Ejemplo

In [11]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

## Requisito

* Implementar una mejora
* Comparar con baseline
* Justificar resultados
* Tabla de resultados
* Análisis breve